# Lab 2: Tools and Short-Term Memory

Lab 1 introduced chat models, message types and the manual management of a conversation history in a text-to-SQL setting. This laboratory extends that setting in three directions:

1. **A locally hosted, open-weight model.** Instead of a hosted API, the model (**Qwen 2.5**, 7 billion parameters) is executed on a GPU through **Ollama**. No API key is required, and no data leaves the execution environment.
2. **Tools.** The model no longer returns an SQL query for the developer to execute. It is given a *tool* that executes the query, and it decides when to call this tool, how to interpret its result and whether to retry after an error. The combination of model, tools and the loop that connects them constitutes an **agent** (see *Anatomy of an agent* in Lab 1).
3. **Short-term memory.** The conversation history, which was maintained manually in Lab 1, is stored by the agent itself, together with additional structured information about the user.

The examples again use the Formula 1 database from the BIRD benchmark. The notebook is designed for **Google Colab with a GPU runtime** (*Runtime → Change runtime type → T4 GPU*). It can also be executed on a local machine with Ollama installed, in which case the Colab-specific installation cells can be skipped.

### Structure of the notebook

| Section | Concepts |
|---|---|
| Setting up the environment | Repository, Python and system dependencies |
| Running a local model with Ollama | Ollama server, Qwen 2.5, `model_provider="ollama"` |
| Tools | `@tool`, tool schemas, read-only database access, error handling |
| The agent loop | `create_agent`, tool calls, `ToolMessage`, self-correction, tool failure |
| Short-term memory | Checkpointer and threads, custom state, state-aware tools, middleware |
| Exercise | A membership assistant for the student club database |

## Setting up the environment

The data used in this laboratory (the Formula 1 and student club databases from BIRD) are distributed through the course repository. The first cell clones the repository; the second cell sets it as the working directory, so that relative paths such as `data/formula_1/formula_1.sqlite` resolve correctly. The check on `Path.cwd().name` makes the second cell safe to execute more than once.

In [ ]:
!git clone https://github.com/iuliabanu/BDNSV_lab1

In [2]:
import os
from pathlib import Path

PROJECT_DIR = "BDNSV_lab1"

if Path.cwd().name != PROJECT_DIR:
    os.chdir(PROJECT_DIR)

print(f"Working directory: {Path.cwd()}")

Working directory: /content/BDNSV_lab1


### Installing the dependencies

Two groups of dependencies are required:

- **Python packages.** `langchain` provides the agent abstractions (and installs `langgraph` as a dependency), `langchain-ollama` provides the integration with Ollama, and `sqlalchemy` provides access to the SQLite databases.
- **System packages.** The Ollama installer distributes its binaries as a `.tar.zst` archive, which requires the `zstd` decompressor. The `pciutils` package provides `lspci`, which the installer uses to detect the GPU and to install the corresponding drivers.

In [ ]:
!pip install -qU langchain langchain-ollama sqlalchemy

In [ ]:
print("Installing zstd extractor and cuda support...")
!sudo apt-get update -y && sudo apt-get install -y zstd pciutils

## Running a local model with Ollama

**Ollama** is an open-source runtime for executing large language models on local hardware. It downloads model weights from a public library, loads them on the GPU (or on the CPU, if no GPU is available), and exposes them through an HTTP API on `http://127.0.0.1:11434`. LangChain communicates with this API through the `langchain-ollama` integration, so that a local model can be used through the same interface as a hosted one.

The following cell performs two steps:

1. **Installation.** The official installation script copies the `ollama` binary to `/usr/local` and detects the NVIDIA GPU.
2. **Starting the server.** Colab does not run `systemd`, so the service configured by the installer is not started automatically (hence the warning `systemd is not running`). The server is therefore started explicitly with `ollama serve` as a background process through `subprocess.Popen`, and its output is redirected to `ollama.log`, which can be inspected if a later step fails. The pause of five seconds gives the server time to start before the first request.

**Hosted and local models compared.**

| Aspect | Hosted API (Lab 1) | Local model (this lab) |
|---|---|---|
| Access | API key, subject to rate limits | No key; limited only by the available hardware |
| Data | Sent to the provider | Remains in the execution environment |
| Model size | Very large models (hundreds of billions of parameters) | Small to medium models that fit in GPU memory |
| Quality | Generally higher | Lower, especially on multi-step reasoning |
| Reproducibility | The provider may update the model | The weights are fixed by the model tag |

In [ ]:
import subprocess
import time

print("Installing Ollama...")
# Use the official Linux installer script provided by Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("Starting Ollama server in the background...")
# Start the server and send logs to a text file
log_file = open("ollama.log", "w")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)
time.sleep(5)
print("Ollama is ready!")


### The Qwen 2.5 model

**Qwen** is a family of open-weight language models developed by the Qwen team at Alibaba Cloud. **Qwen 2.5** (September 2024) comprises models from 0.5 to 72 billion parameters, released mostly under the Apache 2.0 license. This laboratory uses the 7-billion-parameter instruction-tuned model, identified in the Ollama library by the tag `qwen2.5:7b`. The model was selected for the following properties:

- **Size.** The default Ollama build is quantized to 4 bits per weight (a download of about 4.7 GB), which fits comfortably in the 16 GB of memory of a Colab T4 GPU.
- **Tool calling.** The model was trained to produce structured tool calls, a prerequisite for the agents in this laboratory. Not every open-weight model supports this capability.
- **Context window.** The Ollama build supports a context of 32K tokens, sufficient for the database schema, the conversation history and the tool results.

`ollama pull` downloads the model weights once; subsequent calls load them from the local cache. Other models from the Ollama library (for example, more recent Qwen or Llama models with tool support) can be substituted by changing the tag here and in the `init_chat_model` calls below.

A model of this size has limitations that are clearly visible in the outputs of this notebook: it occasionally refers to non-existent columns, miscounts rows and ignores parts of a question. These errors are not defects of the code; they illustrate why the harness, and not only the model, must guarantee the correctness and safety of an agent.

In [6]:
print("Downloading Qwen (this might take a minute)...")
!ollama pull qwen2.5:7b
print("Model pulled successfully!")


Model pulled successfully!


### A first call to the local model

The model is created with `init_chat_model`, exactly as in Lab 1; only the `model` and `model_provider` arguments differ. No API key is needed, and the connection to the Ollama server on `localhost:11434` is established by default (a different server can be specified through the `base_url` parameter). The returned object exposes the same interface (`invoke`, `stream`, `batch`) as the Gemini model used in Lab 1.

In [7]:
from langchain.chat_models import init_chat_model

# Initialize via LangChain's unified interface
llm = init_chat_model(
    model="qwen2.5:7b",
    model_provider="ollama",
    temperature=0.7
)

# Run a test query
response = llm.invoke("What makes Google Colab and Ollama a good combination?")
print(response.content)


Google Colab (Colaboratory) and Ollama are a good combination because they offer complementary functionalities that can enhance the development, testing, and deployment of machine learning models. Here’s a breakdown of how they complement each other:

### 1. **Google Colab (Colaboratory):**
   - **Free Access to GPUs and TPUs:** Google Colab provides access to free GPU and TPU resources, which are essential for training deep learning models efficiently.
   - **Interactive Environment:** It offers a Jupyter notebook environment that allows you to write and execute code, visualize data, and share results.
   - **Automatic Saving:** Colab notebooks are automatically saved to your Google Drive, ensuring you don't lose your work.
   - **Integration with Google Services:** It integrates well with other Google services, such as Google Drive and Google Cloud.

### 2. **Ollama:**
   - **Access to AI Models:** Ollama provides access to a wide range of AI models, including large language models a

**Observation.** The answer is fluent and well structured, but several of its claims about Ollama are inaccurate: Ollama does not provide TPU support, a fine-tuning service or a deployment infrastructure, and it is not a provider of hosted models. The model generates plausible text from its training data rather than from verified sources, a phenomenon known as **hallucination**. Smaller models are particularly prone to it. In the remainder of the laboratory, the model is therefore required to base its answers on data obtained through tools.

## Tools

A **tool** is a function that the model can request to execute. It is described to the model by three elements:

- a **name** (e.g. `run_query`);
- a **description**, which explains what the tool does and when it should be used;
- an **input schema**, which specifies the names and types of the arguments.

The model never executes a tool itself. When it decides that a tool is needed, it returns an `AIMessage` containing one or more **tool calls** (the tool name, the arguments and a call identifier). The application executes the corresponding function and returns the result to the model as a **`ToolMessage`** that carries the same identifier. The model then either requests further tool calls or produces the final answer. This division of responsibilities is fundamental: the model *decides*, the harness *executes*, and it is therefore the harness that controls what a tool is permitted to do.

### A read-only database connection

In this laboratory, the model writes SQL queries that are executed without human review. The connection is therefore opened in **read-only mode** (`mode=ro` in an SQLite URI), so that any `INSERT`, `UPDATE`, `DELETE` or `DROP` statement is rejected by the database engine itself. In Lab 1, `run_sql` only checked whether the query began with `SELECT` or `WITH`; such a textual check is easier to circumvent than a restriction enforced by the database. As a general principle, a safety guarantee should be enforced by the system that executes an action, not requested from the model in the prompt.

In [8]:
from pathlib import Path
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import Engine

DB_PATH = Path("data/formula_1/formula_1.sqlite")
assert DB_PATH.exists(), f"Database not found: {DB_PATH.resolve()}"

engine = create_engine(
    f"sqlite:///file:{DB_PATH.as_posix()}?mode=ro&uri=true",
    connect_args={"uri": True},
)


### Defining tools with `@tool`

The `@tool` decorator from `langchain.tools` converts a Python function into a tool:

- The argument of the decorator (`"run_query"`) sets the **name** of the tool; by default, the name of the function is used.
- The **docstring** becomes the description that the model reads. It must therefore state precisely what the tool does, what it returns and how it behaves on errors.
- The **type hints** (`sql: str`) define the input schema.

Two design decisions in `run_query` deserve attention:

1. **Errors are returned, not raised.** If the query fails, the tool returns the error message as text. The model receives this message as a `ToolMessage` and can correct the query, instead of the agent terminating with an exception.
2. **The output is bounded.** `fetchmany(20)` limits the number of rows, so that a query returning thousands of rows does not exhaust the context window of the model.

`run_query_broken` has the same name, description and schema as `run_query`, but always returns an error. It simulates an unavailable database and is used below to observe how the agent behaves when a tool fails.

**Remark.** The commented line `#@tool(return_direct=True)` shows an alternative option: with `return_direct=True`, the result of the tool is returned directly to the user and the agent loop stops, without a further model call to formulate an answer.

In [10]:
from langchain.tools import tool

#@tool(return_direct=True)
@tool("run_query")
def run_query(sql: str) -> str:
    """Execute a read-only SQLite query on the Formula 1 database.
    Returns the column names and up to 20 rows, or an error message
    if the query fails."""
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            columns = list(result.keys())
            rows = [tuple(row) for row in result.fetchmany(20)]
        return f"Columns: {columns}\nRows ({len(rows)} shown, max 20): {rows}"
    except Exception as error:
        return f"Error: {error}"

@tool("run_query_broken")
def run_query_broken(sql: str) -> str:
    """Execute a read-only SQLite query on the Formula 1 database.
    Returns the column names and up to 20 rows, or an error message
    if the query fails."""
    # Deliberately broken: simulates an unavailable database
    return "Error: database connection unavailable"

### The system prompt

`get_schema` is the same function as in Lab 1. Its output is included in the system prompt, which differs from the one in Lab 1 in an essential respect: the model is no longer asked to *return* a query, but to *obtain data* by calling the tool, to correct the query if the tool reports an error, and to base its answer only on the returned rows. The requirement to include the SQL query in the answer makes the result verifiable by the user.

In [11]:
from langchain.agents import create_agent

def get_schema(engine: Engine) -> str:
    """Return a textual description of all tables, columns and foreign keys."""
    inspector = inspect(engine)
    lines = []
    for table in inspector.get_table_names():
        columns = ", ".join(
            f"{col['name']} {col['type']}" for col in inspector.get_columns(table)
        )
        lines.append(f"{table}({columns})")
        for fk in inspector.get_foreign_keys(table):
            lines.append(
                f"  FOREIGN KEY ({', '.join(fk['constrained_columns'])}) "
                f"REFERENCES {fk['referred_table']}({', '.join(fk['referred_columns'])})"
            )
    return "\n".join(lines)

schema = get_schema(engine)
print(schema)

system_prompt = f"""You are an expert SQLite assistant for a Formula 1 database.
Answer the user's question using the data in the database.

Rules:
- Use only the tables and columns listed in the schema below.
- To obtain data, call the run_query tool with a single read-only (SELECT) query.
- If the tool returns an error, correct the query and call the tool again.
- Base your final answer only on the rows returned by the tool, and include the SQL query you used.

Database schema:
{schema}"""


circuits(circuitId INTEGER, circuitRef TEXT, name TEXT, location TEXT, country TEXT, lat REAL, lng REAL, alt INTEGER, url TEXT)
constructorResults(constructorResultsId INTEGER, raceId INTEGER, constructorId INTEGER, points REAL, status TEXT)
  FOREIGN KEY (raceId) REFERENCES races(raceId)
  FOREIGN KEY (constructorId) REFERENCES constructors(constructorId)
constructorStandings(constructorStandingsId INTEGER, raceId INTEGER, constructorId INTEGER, points REAL, position INTEGER, positionText TEXT, wins INTEGER)
  FOREIGN KEY (raceId) REFERENCES races(raceId)
  FOREIGN KEY (constructorId) REFERENCES constructors(constructorId)
constructors(constructorId INTEGER, constructorRef TEXT, name TEXT, nationality TEXT, url TEXT)
driverStandings(driverStandingsId INTEGER, raceId INTEGER, driverId INTEGER, points REAL, position INTEGER, positionText TEXT, wins INTEGER)
  FOREIGN KEY (raceId) REFERENCES races(raceId)
  FOREIGN KEY (driverId) REFERENCES drivers(driverId)
drivers(driverId INTEGER, dri

## The agent loop: `create_agent`

`create_agent` combines the model, the tools and the system prompt into an agent that executes the following loop (the **ReAct** pattern, *Reasoning and Acting*):

```
user question → model → tool call(s)? ──yes──→ execute tool(s) → ToolMessage(s) → model → ...
                                  └──no───→ final answer
```

The loop continues as long as the model requests tool calls, and terminates when the model returns an `AIMessage` without tool calls. The agent is implemented as a LangGraph graph with two nodes (the model and the tools) and a conditional edge between them.

The input of `invoke()` is a dictionary with a `messages` key; messages may be specified as dictionaries with `role` and `content`. The result is the final **state** of the agent, whose `messages` list contains the complete trace: the question, every tool call, every tool result and the final answer. `pretty_print()` displays each message with its type.

In [12]:
agent = create_agent(
    llm,
    tools=[run_query],
    system_prompt=system_prompt,
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Which driver won the most races in the 2011 season? "
                   "Return the driver's full name and the number of wins.",
    }]
})

# The full trace: question → tool call(s) → tool result(s) → final answer
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Which driver won the most races in the 2011 season? Return the driver's full name and the number of wins.
================================== Ai Message ==================================
Tool Calls:
  run_query (04f2a8aa-703b-4b7e-bdc5-8f9c9c278ceb)
 Call ID: 04f2a8aa-703b-4b7e-bdc5-8f9c9c278ceb
  Args:
    sql: SELECT d.forename, d.surname, COUNT(*) AS wins FROM drivers d JOIN results r ON d.driverId = r.driverId WHERE r.year = 2011 AND r.position = 1 GROUP BY d.driverId ORDER BY wins DESC LIMIT 1;
================================= Tool Message =================================
Name: run_query

Error: (sqlite3.OperationalError) no such column: r.year
[SQL: SELECT d.forename, d.surname, COUNT(*) AS wins FROM drivers d JOIN results r ON d.driverId = r.driverId WHERE r.year = 2011 AND r.position = 1 GROUP BY d.driverId ORDER BY wins DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/21/e3q8)

**Analysis of the trace.**

1. The first query filters on `r.year`, a column that does not exist in the `results` table. The tool returns the SQLite error as a `ToolMessage`.
2. The model reads the error, explains it, and issues a corrected query that joins the `races` table to filter by season.
3. The corrected query returns Sebastian Vettel with 11 wins, which is correct, and the model formulates the final answer.

The self-correction procedure, which had to be programmed explicitly in Exercise 5 of Lab 1, is now performed by the agent loop: the error is simply another observation for the model.

Note, however, that the SQL query shown in the final answer is not identical to the executed one (the alias of `races` is `re` instead of `r2`). The final answer is text generated by the model and not a copy of the tool call. The authoritative record of what was executed is the tool call in the trace.

### Behavior when a tool fails

The following cell creates the same agent with `run_query_broken`, which always reports that the database is unavailable.

In [13]:
agent_broken = create_agent(
    llm,
    tools=[run_query_broken],
    system_prompt=system_prompt,
)

result = agent_broken.invoke({
    "messages": [{
        "role": "user",
        "content": "Which driver won the most races in the 2011 season? "
                   "Return the driver's full name and the number of wins.",
    }]
})

# The full trace: question → tool call(s) → tool result(s) → final answer
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Which driver won the most races in the 2011 season? Return the driver's full name and the number of wins.
================================== Ai Message ==================================
Tool Calls:
  run_query_broken (cf9e9fef-1499-436c-bb97-4720d603deb5)
 Call ID: cf9e9fef-1499-436c-bb97-4720d603deb5
  Args:
    sql: SELECT d.forename, d.surname, COUNT(*) AS wins FROM drivers d JOIN results r ON d.driverId = r.driverId WHERE r.year = 2011 AND r.position = 1 GROUP BY d.driverId ORDER BY wins DESC LIMIT 1;
================================= Tool Message =================================
Name: run_query_broken

Error: database connection unavailable
================================== Ai Message ==================================

Let's try the query again to ensure we get the correct information.

The SQL query to find out which driver won the most races in the 2011 season and their full name along with the

**Analysis of the trace.**

- The model retries the **identical** query three times, although an unavailable connection cannot be remedied by repeating the request. Each retry consumes a model call and adds messages to the context.
- The model then stops and returns the SQL query to the user. This query contains the same error as before (`r.year`), and it was never executed successfully: the answer presents an unverified query as "the correct SQL query".
- On the positive side, the model does not invent a result (it does not state a driver or a number of wins).

The number of retries was decided by the model, not by the application. A less cautious model could continue indefinitely; LangGraph therefore imposes a **recursion limit** on the number of steps of a run (configurable through `recursion_limit` in the `config` argument of `invoke()`), and LangChain provides middleware that limits the number of model or tool calls. Termination and retry policies are thus further examples of guarantees that belong to the harness rather than to the prompt.

## Short-term memory

Chat models are stateless (see Lab 1): each call receives only the messages passed to it. In Lab 1, a follow-up question was supported by manually appending the previous `AIMessage` and the new `HumanMessage` to the message list. An agent created with `create_agent` keeps its data in a **state**, a dictionary that the agent loop reads and updates at every step. This section builds short-term memory in three stages, each of which adds one parameter to `create_agent`:

| Stage | Parameter of `create_agent` | What the agent remembers |
|---|---|---|
| 1 | `checkpointer` | The message history of a conversation |
| 2 | `state_schema` | Structured fields (the identity of the user), readable by the tools |
| 3 | `middleware` | The same fields, now also visible to the model through the system prompt |

### Stage 1: Persisting the conversation with a checkpointer

By default, the state of an agent contains only the list of `messages`, and it is discarded when `invoke()` returns. A **checkpointer** stores a snapshot of the state (a *checkpoint*) after every step. Each conversation is identified by a **`thread_id`**, passed in the `config` argument. A later invocation with the same `thread_id` continues from the stored state, so that only the new message has to be sent. `InMemorySaver` keeps the checkpoints in the memory of the Python process, so they are lost when the kernel restarts; production systems use a database-backed checkpointer instead (for example, `SqliteSaver` or `PostgresSaver`).

The following cell creates a new model instance with `temperature=0`, for more deterministic tool calls, and an agent with an `InMemorySaver`. The helper function `ask` sends one user message to a given thread and prints only the messages produced by this turn: it reads the length of the stored history with `get_state` before the call. Additional keyword arguments are added to the input state; they are used in Stage 2.

In [14]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent_llm = init_chat_model(model="qwen2.5:7b", model_provider="ollama", temperature=0)

sql_agent = create_agent(
    agent_llm,
    tools=[run_query],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)


def ask(agent, question: str, thread_id: str, **state_fields) -> None:
    """Send one user turn to a thread and print only the new messages."""
    config = {"configurable": {"thread_id": thread_id}}
    before = len(agent.get_state(config).values.get("messages", []))
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}], **state_fields},
        config,
    )
    for message in result["messages"][before:]:
        message.pretty_print()


ask(sql_agent, "Which driver won the most races in the 2011 season?", thread_id="f1-1")

================================ Human Message =================================

Which driver won the most races in the 2011 season?
================================== Ai Message ==================================
Tool Calls:
  run_query (6b64cb86-6060-4c89-8af1-071e6ea767e9)
 Call ID: 6b64cb86-6060-4c89-8af1-071e6ea767e9
  Args:
    sql: SELECT d.surname, d.forename FROM drivers d JOIN driverStandings ds ON d.driverId = ds.driverId JOIN races r ON ds.raceId = r.raceId WHERE r.year = 2011 GROUP BY d.driverId ORDER BY SUM(ds.wins) DESC LIMIT 1;
================================= Tool Message =================================
Name: run_query

Columns: ['surname', 'forename']
Rows (1 shown, max 20): [('Vettel', 'Sebastian')]
================================== Ai Message ==================================

The driver who won the most races in the 2011 season was Sebastian Vettel.


In [15]:
# Same thread: "he" and "that season" are resolved from the stored history
ask(sql_agent, "Which constructor did he drive for that season?", thread_id="f1-1")

================================ Human Message =================================

Which constructor did he drive for that season?
================================== Ai Message ==================================
Tool Calls:
  run_query (becc3842-9916-46fe-9175-809c958a0f1a)
 Call ID: becc3842-9916-46fe-9175-809c958a0f1a
  Args:
    sql: SELECT c.name FROM constructors c JOIN results r ON c.constructorId = r.constructorId JOIN races r2 ON r.raceId = r2.raceId JOIN driverStandings ds ON r.driverId = ds.driverId WHERE r2.year = 2011 AND ds.driverId = (SELECT driverId FROM driverStandings WHERE surname = 'Vettel' AND forename = 'Sebastian' AND year = 2011);
================================= Tool Message =================================
Name: run_query

Error: (sqlite3.OperationalError) no such column: surname
[SQL: SELECT c.name FROM constructors c JOIN results r ON c.constructorId = r.constructorId JOIN races r2 ON r.raceId = r2.raceId JOIN driverStandings ds ON r.driverId = ds.driverId W

The second call sends only the follow-up question; the checkpointer supplies the earlier messages. This is the manual `messages.append(...)` of Lab 1, now performed by the agent.

The trace shows that the memory is effective: the model resolves "he" to Sebastian Vettel and "that season" to 2011, information that is present only in the stored history. The task itself, however, is not completed. The model repeatedly refers to columns that do not exist in the `driverStandings` table (`surname`, `forename`, `year`), and the run ends without an answer, although the correct one (Red Bull) requires a simple join of `results`, `races` and `constructors`. Memory and reasoning are independent capabilities: the checkpointer provides the context, but the quality of the queries depends on the model.

The next cell sends the same follow-up question to a new `thread_id`, which starts with an empty history.

In [16]:
# New thread: no stored history, so "he" cannot be resolved
ask(sql_agent, "Which constructor did he drive for that season?", thread_id="f1-2")

================================ Human Message =================================

Which constructor did he drive for that season?
================================== Ai Message ==================================

I need more information to answer your question. Could you please specify the driver and the season you are interested in?


In the new thread, the pronoun "he" cannot be resolved, and the model correctly asks for clarification instead of guessing. Each thread is therefore an isolated conversation.

In this stage, the memory consists only of messages. Any fact about the user, such as their identity, is available only as text inside those messages, and tools cannot access it reliably.

### Stage 2: Extending the state with a custom schema

The scenario now changes: the agent becomes a personal assistant for a Formula 1 driver who asks about the driver's own results. The identity of the driver is stored in the state as structured fields rather than as text.

- **`state_schema`.** `DriverState` extends `AgentState` with `driver_id`, `forename` and `surname`. These fields are provided in the first invocation of a thread and persisted by the checkpointer together with the messages.
- **State-aware tools.** A tool can declare a `runtime: ToolRuntime` parameter to read the state through `runtime.state`. This parameter is injected by the agent and hidden from the model: it does not appear in the tool schema. The tools below read `driver_id` from the state, so that the model can only access the results of the identified driver.

Unlike `run_query`, these tools do not accept SQL from the model. Each executes a fixed, **parameterized** query (`:driver_id`, `:year`), in which the values are passed separately from the SQL text and therefore cannot alter its structure. The model chooses only the tool and the optional season. Restricting the actions available to the model in this way is a stronger guarantee than the rule "Do not answer questions about other drivers" in the system prompt.

In [17]:
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime


class DriverState(AgentState):
    """Agent state extended with the identity of the current driver."""
    driver_id: int
    forename: str
    surname: str


def query_rows(sql: str, params: dict) -> tuple[list[str], list[tuple]]:
    """Execute a parameterized read-only query and return column names and rows."""
    with engine.connect() as conn:
        result = conn.execute(text(sql), params)
        return list(result.keys()), [tuple(row) for row in result]


@tool
def get_my_results(runtime: ToolRuntime, year: int | None = None) -> str:
    """Return the race results of the current driver: season, round, race name,
    constructor, grid position, finishing position, points and status.
    Optionally filter by season (year). Returns at most 30 rows."""
    driver_id = runtime.state.get("driver_id")
    if driver_id is None:
        return "Error: no driver is identified in this conversation."
    columns, rows = query_rows(
        """
        SELECT ra.year, ra.round, ra.name AS race, c.name AS constructor,
               r.grid, r.positionText AS position, r.points, s.status
        FROM results r
        JOIN races ra        ON r.raceId = ra.raceId
        JOIN constructors c  ON r.constructorId = c.constructorId
        JOIN status s        ON r.statusId = s.statusId
        WHERE r.driverId = :driver_id
          AND (:year IS NULL OR ra.year = :year)
        ORDER BY ra.year, ra.round
        """,
        {"driver_id": driver_id, "year": year},
    )
    return f"Columns: {columns}\nRows ({len(rows)} total, max 30 shown): {rows[:30]}"


@tool
def get_my_season_summary(runtime: ToolRuntime, year: int) -> str:
    """Return a summary of the current driver's season: number of races,
    wins, podiums, starts from first place on the grid, and total points."""
    driver_id = runtime.state.get("driver_id")
    if driver_id is None:
        return "Error: no driver is identified in this conversation."
    columns, rows = query_rows(
        """
        SELECT COUNT(*)              AS races,
               SUM(r.position = 1)   AS wins,
               SUM(r.position <= 3)  AS podiums,
               SUM(r.grid = 1)       AS grid_first,
               SUM(r.points)         AS points
        FROM results r
        JOIN races ra ON r.raceId = ra.raceId
        WHERE r.driverId = :driver_id AND ra.year = :year
        """,
        {"driver_id": driver_id, "year": year},
    )
    return f"Columns: {columns}\nRows: {rows}"


driver_system_prompt = """You are a personal assistant for a Formula 1 driver.

Rules:
- Answer questions about the driver's own results, using only the tools.
- Do not answer questions about other drivers.
- Base your answer only on the data returned by the tools.
- Address the driver directly ("you won ...")."""

driver_agent = create_agent(
    agent_llm,
    tools=[get_my_results, get_my_season_summary],
    system_prompt=driver_system_prompt,
    state_schema=DriverState,
    checkpointer=InMemorySaver(),
)

The identifier of the driver is obtained from the database through the stable reference `driverRef` rather than hard-coded. The `ask` helper passes `driver_id`, `forename` and `surname` as additional fields of the input state; the checkpointer stores them with the thread, so they need to be supplied only in the first invocation.

In [18]:
# Look up the driver instead of hard-coding the id
_, rows = query_rows(
    "SELECT driverId, forename, surname FROM drivers WHERE driverRef = :ref",
    {"ref": "vettel"},
)
driver_id, forename, surname = rows[0]

# The identity is passed only in the first invocation of the thread
ask(driver_agent, "How did my 2011 season go?", thread_id="vettel-1",
    driver_id=driver_id, forename=forename, surname=surname)

================================ Human Message =================================

How did my 2011 season go?
================================== Ai Message ==================================
Tool Calls:
  get_my_season_summary (0de3d713-a2b4-4868-ade0-332a6edf81da)
 Call ID: 0de3d713-a2b4-4868-ade0-332a6edf81da
  Args:
    year: 2011
================================= Tool Message =================================
Name: get_my_season_summary

Columns: ['races', 'wins', 'podiums', 'grid_first', 'points']
Rows: [(19, 11, 17, 15, 392.0)]
================================== Ai Message ==================================

In the 2011 season, you participated in 19 races. You won 11 of them, secured 17 podiums, started 15 races from the pole position, and accumulated a total of 392 points.


**Observation.** The model selects `get_my_season_summary` and passes only the season; the identity of the driver is supplied by the state. The figures in the answer (19 races, 11 wins, 17 podiums, 15 starts from the first grid position, 392 points) match the tool result.

In [19]:
# Same thread: neither the identity nor the season is repeated
ask(driver_agent, "In which races that season did I start first on the grid but not win?",
    thread_id="vettel-1")

state = driver_agent.get_state({"configurable": {"thread_id": "vettel-1"}}).values
print(f"\nStored state: driver_id={state['driver_id']}, "
      f"name={state['forename']} {state['surname']}, messages={len(state['messages'])}")

================================ Human Message =================================

In which races that season did I start first on the grid but not win?
================================== Ai Message ==================================
Tool Calls:
  get_my_results (9fd34f2d-05e5-4ab0-9185-e4d252d6d097)
 Call ID: 9fd34f2d-05e5-4ab0-9185-e4d252d6d097
  Args:
    year: 2011
================================= Tool Message =================================
Name: get_my_results

Columns: ['year', 'round', 'race', 'constructor', 'grid', 'position', 'points', 'status']
Rows (19 total, max 30 shown): [(2011, 1, 'Australian Grand Prix', 'Red Bull', 1, '1', 25.0, 'Finished'), (2011, 2, 'Malaysian Grand Prix', 'Red Bull', 1, '1', 25.0, 'Finished'), (2011, 3, 'Chinese Grand Prix', 'Red Bull', 1, '2', 18.0, 'Finished'), (2011, 4, 'Turkish Grand Prix', 'Red Bull', 1, '1', 25.0, 'Finished'), (2011, 5, 'Spanish Grand Prix', 'Red Bull', 2, '1', 25.0, 'Finished'), (2011, 6, 'Monaco Grand Prix', 'Red Bull', 1

**Observation.** The follow-up question relies on the memory twice: the identity of the driver comes from the state, and "that season" is resolved from the message history. The stored state confirms that the identity has been persisted, together with 8 messages (two turns of four messages each: question, tool call, tool result, answer).

The answer, however, is **incorrect**, although the tool returned the correct rows. Of the four races listed, only the Canadian Grand Prix satisfies the condition: Vettel started second in the British and Korean Grands Prix, and he won the Korean and Indian Grands Prix. The correct answer consists of six races: China, Canada, Hungary, Japan, Abu Dhabi (retirement due to a puncture) and Brazil. The statement that all of them ended on the podium is also false for Abu Dhabi.

Filtering and counting over rows are operations at which language models, and small models in particular, are unreliable. Such operations should preferably be delegated to a tool, for example a query with the condition `grid = 1 AND position <> 1`.

The next cell examines whether the name stored in the state is available to the model.

In [20]:
# The name is in the state, but is it visible to the model?
ask(driver_agent, "What is my name?", thread_id="vettel-1")

================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

To provide you with the most accurate information, could you please tell me your name or check it from your recent race results? If you're not sure, I can fetch your name from the 2011 season results. Would you like me to do that?


**Observation.** The model cannot answer, although the name is stored in the state. The model perceives only the system prompt, the messages and the tool schemas; the custom fields of the state are accessible to the tools, but not to the model. The answer also illustrates a further risk: the model offers to retrieve the name from the race results, which `get_my_results` does not return.

### Stage 3: Building the prompt from the state with middleware

**Middleware** intercepts the agent loop at defined points and can inspect or modify what the model receives and returns. LangChain provides hooks that run before and after each model call (`before_model`, `after_model`) and hooks that wrap each model or tool call (`wrap_model_call`, `wrap_tool_call`); typical applications include dynamic prompts, summarization of long histories, limits on the number of calls and human approval of tool calls.

The `@dynamic_prompt` decorator is a shorthand for the most common case: it defines a middleware that builds the system prompt before every model call, with access to the current state through `request.state`. The static `system_prompt` argument is therefore omitted, and the function `driver_prompt` prepends the name of the driver to `driver_system_prompt`.

In [21]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dynamic_prompt
def driver_prompt(request: ModelRequest) -> str:
    """Build the system prompt from the identity stored in the state."""
    state = request.state
    name = f"{state.get('forename', '')} {state.get('surname', '')}".strip()
    return f"""You are talking to: {name or "an unidentified user"}.

{driver_system_prompt}"""


driver_agent = create_agent(
    agent_llm,
    tools=[get_my_results, get_my_season_summary],
    state_schema=DriverState,
    middleware=[driver_prompt],
    checkpointer=InMemorySaver(),
)

ask(driver_agent, "What is my name, and how many races did I win in 2011?",
    thread_id="vettel-3", driver_id=driver_id, forename=forename, surname=surname)

================================ Human Message =================================

What is my name, and how many races did I win in 2011?
================================== Ai Message ==================================
Tool Calls:
  get_my_results (8e1f1066-7c0c-47ea-9f20-2bb59f455196)
 Call ID: 8e1f1066-7c0c-47ea-9f20-2bb59f455196
  Args:
    year: 2011
================================= Tool Message =================================
Name: get_my_results

Columns: ['year', 'round', 'race', 'constructor', 'grid', 'position', 'points', 'status']
Rows (19 total, max 30 shown): [(2011, 1, 'Australian Grand Prix', 'Red Bull', 1, '1', 25.0, 'Finished'), (2011, 2, 'Malaysian Grand Prix', 'Red Bull', 1, '1', 25.0, 'Finished'), (2011, 3, 'Chinese Grand Prix', 'Red Bull', 1, '2', 18.0, 'Finished'), (2011, 4, 'Turkish Grand Prix', 'Red Bull', 1, '1', 25.0, 'Finished'), (2011, 5, 'Spanish Grand Prix', 'Red Bull', 2, '1', 25.0, 'Finished'), (2011, 6, 'Monaco Grand Prix', 'Red Bull', 1, '1', 25.0, 'F

**Observation.** The name of the driver is now part of the system prompt. In this run, however, the model answers only the second part of the question: it does not state the name, and it reports 9 wins, whereas the rows returned by the tool contain 11 wins. The model selected `get_my_results` and counted the wins itself instead of using `get_my_season_summary`, which computes the number in SQL. Multi-part questions and counting over tool output are typical weaknesses of small models; it is instructive to re-run the cell or to ask only "What is my name?".

In [22]:
# A new thread has an empty state: no identity, no history
ask(driver_agent, "How did my 2011 season go?", thread_id="vettel-4")

================================ Human Message =================================

How did my 2011 season go?
================================== Ai Message ==================================
Tool Calls:
  get_my_season_summary (4151002d-fa84-4254-a0ed-e88b7604ff7b)
 Call ID: 4151002d-fa84-4254-a0ed-e88b7604ff7b
  Args:
    year: 2011
================================= Tool Message =================================
Name: get_my_season_summary

Error: no driver is identified in this conversation.
================================== Ai Message ==================================

It seems there was an issue identifying your driver details. Could you please provide more information about which driver you are referring to?


The middleware places the name of the driver in the system prompt before every model call, so that the information stored in the state is now available to the model as well as to the tools. In a new thread the state is empty: the prompt reports an unidentified user, and the tools return an error instead of data.

Short-term memory is therefore scoped to a single thread. Information that must persist *across* threads, such as preferences or facts learned in earlier conversations, requires **long-term memory**, which LangGraph implements through a *store* organized by namespaces (for example, per user).

**Remark: state vs. runtime context.** In production, the identity of the logged-in user is often passed as *runtime context* (`context_schema`), because it is fixed by the application and should not change during a run. Here it is stored in the state, since the subject of this section is memory: the identity is persisted with the thread and remains available across turns.

### Summary of the key concepts

| Concept | Role in this laboratory |
|---|---|
| Ollama / Qwen 2.5 | Local execution of an open-weight model with tool-calling support |
| Tool (`@tool`) | A function with a name, description and input schema that the model can request to execute |
| Tool call / `ToolMessage` | The model's request to execute a tool, and the result returned to the model |
| Agent loop (`create_agent`) | Model → tool calls → tool results → model, until a final answer is produced |
| State | The data of a run: the messages and, optionally, custom fields |
| Checkpointer and `thread_id` | Persistence of the state between invocations of the same conversation |
| `state_schema` and `ToolRuntime` | Structured information in the state, readable by tools but hidden from the model |
| Middleware (`@dynamic_prompt`) | Modification of the model input, for example a system prompt built from the state |

## Exercise: A membership assistant for the student club

*Objective: to design an agent whose state identifies the user and whose tools are scoped to that user.*

The database `data/student_club/student_club.sqlite` contains the members of a student club, the events they attended, their payments (income) and the expenses they submitted. The tables relevant to a single member include `member`, `major`, `attendance`, `event`, `income`, `expense` and `budget`.

1. Create a read-only engine for the student club database, as for the Formula 1 database. Inspect the schema with `get_schema` and the column descriptions with `load_column_descriptions` (defined in Lab 1; copy the function into this notebook). Identify the tables and foreign keys that relate to a single member.
2. Define a `MemberState` that extends `AgentState` with `member_id`, `first_name` and `last_name`. Note that member identifiers in this database are text values, not integers.
3. Implement at least three tools that read `member_id` from the state through `ToolRuntime`, for example:
   - the member's profile (position, major, contact data);
   - the events the member attended;
   - the member's payments and submitted expenses.

   Use parameterized queries. The model must not be able to choose the `member_id`, and the agent must not be given a general-purpose tool such as `run_query`.
4. Write a `@dynamic_prompt` that addresses the member by name.
5. Look up a member in the database (as was done for the driver), create the agent with an `InMemorySaver` and test a conversation of at least three turns in one thread. At least one follow-up question must rely on an earlier turn (e.g. "How much did I spend at *that* event?"; observe that expenses are linked to events through the `budget` table).
6. Start a second thread for a different member. Verify that the two conversations share neither the identity nor the history.
7. Ask the agent about another member (e.g. "Which events did <another member's name> attend?"). Does the agent refuse, or does it leak data? Explain which part of your design prevents the leak, and which part only asks the model not to answer. Explain also why the presence of a `run_query` tool would invalidate this guarantee.
8. For each answer, compare the figures stated by the model with the rows returned by the tools (as in the analysis of Stage 2). Report any discrepancy and, where possible, move the computation into a tool.

## References

**Official documentation**
- LangChain. *Agents*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/agents
- LangChain. *Models*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/models
- LangChain. *Tools*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/tools
- LangChain. *Short-term memory*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/short-term-memory
- LangChain. *Long-term memory*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/long-term-memory
- LangChain. *Middleware*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/middleware
- LangChain. *Runtime*. LangChain Documentation. https://docs.langchain.com/oss/python/langchain/runtime
- LangChain. *Persistence*. LangGraph Documentation. https://docs.langchain.com/oss/python/langgraph/persistence
- LangChain. *ChatOllama*. LangChain Integrations. https://docs.langchain.com/oss/python/integrations/chat/ollama
- Ollama. *Ollama Documentation*. https://docs.ollama.com (in particular *Linux* installation: https://docs.ollama.com/linux, and *Tool calling*: https://docs.ollama.com/capabilities/tool-calling)
- Ollama. *qwen2.5*. Ollama Model Library. https://ollama.com/library/qwen2.5

**Source code repositories**
- LangChain: https://github.com/langchain-ai/langchain
- LangGraph: https://github.com/langchain-ai/langgraph
- Ollama: https://github.com/ollama/ollama
- Qwen 2.5: https://github.com/QwenLM/Qwen2.5

**Courses and articles**
- Qwen Team (2024). *Qwen2.5: A Party of Foundation Models!* Qwen Blog. https://qwenlm.github.io/blog/qwen2.5/
- LangChain Academy. *Introduction to LangGraph*. Online course. https://academy.langchain.com/courses/intro-to-langgraph
- Schluntz, E., & Zhang, B. (2024). *Building Effective Agents*. Anthropic Engineering Blog. https://www.anthropic.com/engineering/building-effective-agents

**Research papers**
- Qwen Team (2024). *Qwen2.5 Technical Report*. arXiv preprint arXiv:2412.15115. https://arxiv.org/abs/2412.15115
- Schick, T., Dwivedi-Yu, J., Dessì, R., Raileanu, R., Lomeli, M., Zettlemoyer, L., Cancedda, N., & Scialom, T. (2023). *Toolformer: Language Models Can Teach Themselves to Use Tools*. Advances in Neural Information Processing Systems (NeurIPS). https://arxiv.org/abs/2302.04761
- Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models*. International Conference on Learning Representations (ICLR). https://arxiv.org/abs/2210.03629
- Li, J., Hui, B., Qu, G., et al. (2023). *Can LLM Already Serve as A Database Interface? A BIg Bench for Large-Scale Database Grounded Text-to-SQLs* (BIRD). Advances in Neural Information Processing Systems (NeurIPS), Datasets and Benchmarks Track. https://arxiv.org/abs/2305.03111 (source of the Formula 1 and student club databases)


> **Note on the preparation of this document.** Parts of this laboratory guide were drafted with the assistance of Claude (Anthropic), an AI assistant, used through Claude Code.